# Forecast initial-shock decay timescales

Regional ensemble-mean differences between a shock experiment and a baseline hindcast from the same start dates, with an exponential fit `|shock - baseline| = a exp(-lead/tau) + c` giving the decay timescale of the initialization shock. Per-experiment shock diagnostics against observations are in `01_compute_and_plot_ens_ts.ipynb`.

Cases: **S0** hindcasts start from the Dec 2011 analyses and cover Jan-Feb 2012; **S1** hindcasts start from the May 2012 analyses and cover Jun-Jul 2012. Each case compares hindcasts initialized from CTRL, CAPT and EAM-DART states.

## 1. Settings

In [ ]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
workflow_name = "fcst_atm_initial_shock"

run_segment = "fc"
# season -> (shock experiment, baseline experiment) pairs
comparisons = {
    "S0": (("CAPT10-S0", "CTRL10-S0"),),
    "S1": (("CAPT10-S1", "CTRL10-S1"),),
}
variables = ("TS", "TREFHT", "PSL", "PRECT")
regions = ("global",)
max_members = None  # e.g. 2 for a quick test
force_compute = False
show = True
FIGURE_DPI = 200

## 2. Project setup

In [ ]:
import sys
import matplotlib.pyplot as plt

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.bias_metrics import experiment_registry
from util.forecast_regional_timeseries import REGIONS, read_model_regional_series
from util.forecast_initial_shock import cache_shock_decay, plot_shock_decay, shock_cache_path

data_dir, figure_dir = workflow_output_dirs(
    workflow_name, output_root=DIAGNOSTIC_OUTPUT_ROOT
)
unknown = set(regions) - set(REGIONS)
if unknown:
    raise ValueError(f"Unknown regions {sorted(unknown)}; choose from {list(REGIONS)}")

## 3. Compute or reuse decay fits

In [ ]:
processed = {}
for season, pairs in comparisons.items():
    registry = experiment_registry(INPUT_DATA_ROOT, season, run_segment)
    for shock_experiment, baseline_experiment in pairs:
        period = registry[shock_experiment]["period"]
        if registry[baseline_experiment]["period"] != period:
            raise ValueError(f"{shock_experiment} and {baseline_experiment} periods differ")
        for variable in variables:
            for region in regions:
                target = shock_cache_path(
                    data_dir, variable, region, shock_experiment, baseline_experiment, period,
                )
                if target.is_file() and not force_compute:
                    ds = cache_shock_decay(target)
                    print("[REUSE]", target)
                else:
                    shock, baseline = (
                        read_model_regional_series(
                            INPUT_DATA_ROOT, experiment, season, run_segment,
                            variable, region, period, max_members=max_members,
                        )[0]
                        for experiment in (shock_experiment, baseline_experiment)
                    )
                    ds = cache_shock_decay(
                        target, shock=shock, baseline=baseline, force_compute=True,
                        shock_experiment=shock_experiment,
                        baseline_experiment=baseline_experiment,
                        variable=variable, region=region,
                    )
                    print("[WRITE]", target)
                processed[(variable, region, shock_experiment, baseline_experiment)] = ds

## 4. Plot, save, and show

In [ ]:
for (variable, region, shock_experiment, baseline_experiment), ds in processed.items():
    fig = plot_shock_decay(ds)
    output = figure_dir / (
        f"{variable}_{region}_{shock_experiment}_minus_{baseline_experiment}_shock_decay.png"
    )
    fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"tau = {float(ds.tau_days):.2f} days", "[FIGURE]", output)
    if not show:
        plt.close(fig)

## Notes: variables, regions, and cached fields

Variables: `TS`, `TREFHT`, `PSL`, and `PRECT`. Regions: `global`, `CONUS`, `NHMidLat`, `Tropics`, `NINO3.4`, and `MAR_CONT`.

Caches: `<variable>_<region>_<shock>_minus_<baseline>_<period>_shock_decay.nc`, holding the signed and absolute anomalies, the fitted magnitude, the ensemble-mean `tau_days`, and member-level decay times.